## **Binary Data-preprocessing**


In [10]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split

from config import (
    CSV_PATH,
    XRAY_DIR,
    OUT_DIR,
)


def load_data(csv_path):
    df = pd.read_csv(csv_path)
    rename_map = {
        "Image Index": "IMGPATH",
        "Finding Labels": "DISEASELABEL",
        "Follow-up #": "FOLLOWUP",
        "Patient ID": "PATID",
        "Patient Age": "AGE",
        "Patient Gender": "GENDER",
        "View Position": "VP",
    }
    df = df.rename(columns=rename_map)
    return df


def filter_main_diseases(df):
    main_diseases = {
        "Atelectasis",
        "Cardiomegaly",
        "Emphysema",
        "Fibrosis",
        "Effusion",
        "Mass",
        "Nodule",
        "Pneumonia",
        "Pneumothorax",
        "Consolidation",
        "Edema",
        "No Finding",
    }
    df = df[df["DISEASELABEL"].isin(main_diseases)]
    return df


def filter_last_followups(df):
    df = df.sort_values("FOLLOWUP").groupby("PATID").tail(3)
    return df


def extract_disease_category(df, root_folder):
    df = df.copy()
    df["HOTLABEL"] = df["DISEASELABEL"].apply(lambda x: 1 if x != "No Finding" else 0)

    def find_full_path(image_name):
        for subdir in os.listdir(root_folder):
            full_path = os.path.join(root_folder, subdir, image_name)
            if os.path.exists(full_path):
                return full_path
        return None

    df["IMGPATH"] = df["IMGPATH"].apply(find_full_path)
    
    return df[
        [
            "IMGPATH",
            "DISEASELABEL",
            "HOTLABEL",
            "FOLLOWUP",
            "PATID",
            "AGE",
            "GENDER",
            "VP",
        ]
    ]


def split_dataset(df, test_size=0.15, val_size=0.15, inferno_size=0.05, seed=2025):
    train_val, test = train_test_split(
        df, test_size=test_size, stratify=df["DISEASELABEL"], random_state=seed
    )
    train, val = train_test_split(
        train_val,
        test_size=val_size / (1 - test_size),
        stratify=train_val["DISEASELABEL"],
        random_state=seed,
    )
    train, inferno = train_test_split(
        train,
        test_size=inferno_size / (1 - test_size - val_size),
        stratify=train["DISEASELABEL"],
        random_state=seed,
    )
    
    return train, val, test, inferno


def preprocess_data(csv_path, root_folder):
    df = load_data(csv_path)
    df = filter_main_diseases(df)
    df = filter_last_followups(df)
    df = extract_disease_category(df, root_folder)
    print(len(df))
    train, val, test, inferno = split_dataset(df)
    return train, val, test, inferno


train_df, val_df, test_df, inferno_df = preprocess_data(CSV_PATH, XRAY_DIR)
train_df.to_csv(OUT_DIR / "train.csv", index=False)
val_df.to_csv(OUT_DIR / "val.csv", index=False)
test_df.to_csv(OUT_DIR / "test.csv", index=False)
inferno_df.to_csv(OUT_DIR / "inferno.csv", index=False)

46704


## **Test Dataloader**


In [ ]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from PIL import Image
import matplotlib.pyplot as plt

from config import (
    OUT_DIR,
)

import os
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from PIL import Image
import matplotlib.pyplot as plt


# Custom Dataset Class
class ChestXRayDataset(Dataset):
    def __init__(self, csv_file, transform=None):
        self.data = pd.read_csv(csv_file)
        self.transform = transform

        # Define diseases that should receive augmentation
        self.augmented_diseases = {"Emphysema", "Fibrosis", "Hernia"}

        # Define transformation for selected diseases
        self.augmentation = T.Compose(
            [
                T.RandomRotation(degrees=10),
                T.RandomAffine(degrees=0, translate=(0.05, 0.05)),
                T.GaussianBlur(kernel_size=3, sigma=(0.1, 0.3)),
            ]
        )

        # Standard preprocessing for all images
        self.base_transform = T.Compose([T.ToTensor()])

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_path = self.data.iloc[idx]["IMGPATH"]
        image = Image.open(img_path).convert("L")  # Convert to grayscale
        label = torch.tensor(self.data.iloc[idx]["HOTLABEL"], dtype=torch.long)

        # Store original image before any transformation
        original_image = image.copy()
        print(img_path)
        print(self.data.iloc[idx]["DISEASELABEL"])
        # Apply augmentation only to selected Structural Abnormalities
        if self.data.iloc[idx]["DISEASELABEL"] in self.augmented_diseases:
            image = self.augmentation(image)

        # Apply base transformation to all images
        image = self.base_transform(image)
        original_image = self.base_transform(original_image)

        return original_image, image, label


# DataLoader Wrapper
def get_dataloader(csv_file, batch_size=32, shuffle=True, num_workers=4):
    dataset = ChestXRayDataset(csv_file)
    return DataLoader(
        dataset, batch_size=batch_size, shuffle=shuffle, num_workers=num_workers
    )


# Test Function to Visualize Augmentations
def visualize_augmentation(csv_file, num_samples=5):
    dataset = ChestXRayDataset(csv_file)

    fig, axes = plt.subplots(num_samples, 2, figsize=(8, num_samples * 4))
    for i in range(num_samples):
        idx = torch.randint(len(dataset), size=(1,)).item()
        original_image, augmented_image, label = dataset[idx]

        # Convert tensor image back to PIL format for visualization
        original_pil = T.ToPILImage()(original_image)
        augmented_pil = T.ToPILImage()(augmented_image)

        axes[i, 0].imshow(original_pil, cmap="gray")
        axes[i, 0].set_title(f"Original Image - Label: {label.item()}")
        axes[i, 0].axis("off")

        axes[i, 1].imshow(augmented_pil, cmap="gray")
        axes[i, 1].set_title(f"Augmented Image - Label: {label.item()}")
        axes[i, 1].axis("off")

    plt.tight_layout()
    plt.show()


# Example Usage
train_loader = get_dataloader(OUT_DIR / "train.csv", batch_size=32)
visualize_augmentation(OUT_DIR / "train.csv", num_samples=5)

In [ ]:
from src.Dataset import ChestXRayDataset
import os
import pandas as pd
import torch
from torch.utils.data import Dataset
import torchvision.transforms as T
from PIL import Image
import matplotlib.pyplot as plt


from config import (
    OUT_DIR,
)

dataset = ChestXRayDataset(OUT_DIR / "train.csv")
sample_image, sample_label = dataset[1081]

sample_image_pil = T.ToPILImage()(sample_image)

plt.imshow(sample_image_pil, cmap="gray")
plt.title(f"Augmented Image - Label: {sample_label.item()}")
plt.axis("off")
plt.show()

In [ ]:
import numpy as np
from PIL import Image

img_path = "/home/maxvill/InfernoCalibNet/data/raw/xrays/xrays5/00016933_000.png"  # Change this to your image path
image = Image.open(img_path).convert("L")  # Load grayscale
image_np = np.array(image)

print(f"Min pixel value: {image_np.min()}, Max pixel value: {image_np.max()}")